<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%ED%8F%90%EA%B8%B0)_RAGAS_tutorial_1_(%EC%A0%95%EC%8B%9D%ED%8A%9C%ED%86%A0%EB%A6%AC%EC%96%BC_%EC%8B%A4%ED%96%89_%EB%B6%88%EA%B0%80%EB%8A%A5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#환경 설정

In [ ]:
!pip install ragas

In [ ]:
!ragas quickstart rag_eval

In [ ]:
!pip install -e rag_eval

In [2]:
!pip install "ragas[examples]"

In [1]:
import os
os.environ['OPENAI_API_KEY'] = ''

#Prompt Evaluation

examples 의존성 그룹 수동 다운로드, 수동 실행

In [2]:
!pip install ragas[examples]

In [1]:
!git clone https://github.com/vibrantlabsai/ragas.git
%cd ragas
!pip install -e '.[examples]'

fatal: destination path 'ragas' already exists and is not an empty directory.
/content/ragas
Obtaining file:///content/ragas
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ragas (pyproject.toml) ... done
  Created wheel for ragas: filename=ragas-0.4.4.dev8+g298b68274-0.editable-py3-none-any.whl size=13353 sha256=e611ec3f87a439ce23081cb33c25789a962d151a63c84010c1be28aeca6358c3
  Stored in directory: /tmp/pip-ephem-wheel-cache-2ii20jtn/wheels/8c/b7/75/727c7e0e7207f46773c9ca04e1abb11ddba7e0ba53e63ce785
Successfully built ragas
  Attempting uninstall: ragas
    Found existing installation: ragas 0.4.4.dev8+g298b68274
    Uninstalling ragas-0.4.4.dev8+g298b68274:
      Successfully uninstalled ragas-0.4.4.dev8+g298b68274


In [24]:
!python /content/ragas/examples/ragas_examples/prompt_evals/prompt.py

positive


데이터 만들기

In [5]:
import pandas as pd
import os
samples = [{"text": "I loved the movie! It was fantastic.", "label": "positive"},
    {"text": "The movie was terrible and boring.", "label": "negative"},
    {"text": "It was an average film, nothing special.", "label": "positive"},
    {"text": "Absolutely amazing! Best movie of the year.", "label": "positive"}]

# os.mkdir('datasets')
pd.DataFrame(samples).to_csv("datasets/test_dataset.csv", index=False)

In [7]:
data = pd.read_csv('datasets/test_dataset.csv')
display(data)

,text,label
0,I loved the movie! It was fantastic.,positive
1,The movie was terrible and boring.,negative
2,"It was an average film, nothing special.",positive
3,Absolutely amazing! Best movie of the year.,positive


metric 정의

In [9]:
from ragas.metrics import discrete_metric
from ragas.metrics.result import MetricResult

@discrete_metric(name = 'accuracy', allowed_value = ['pass','fail'])
def my_metric(prediction : str, actual : str):
  return MetricResult(value = 'pass', reason = '') if prediction == actual else MetricResult(value='fail', reason='')

In [18]:
from ragas import experiment

import sys
sys.path.insert(0, '/content/ragas/examples')
from ragas_examples.prompt_evals.prompt import run_prompt

@experiment()
async def run_experiment(row):

  response = run_prompt(row['text'])
  score = my_metric.score(
      prediction=response,
      actual=row['label']
  )

  experiment_view = {
      **row,
      'response' : response,
      'score' : score.value
  }

  return experiment_view

ModuleNotFoundError: No module named 'ragas_examples._version'

In [13]:
run_experiment.arun(data, 'gpt-4-mini')

<coroutine object ExperimentWrapper.arun at 0x7941a9084b40>

In [3]:
!git clone https://github.com/vibrantlabsai/ragas.git

Cloning into 'ragas'...
remote: Enumerating objects: 12804, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 12804 (delta 149), reused 85 (delta 85), pack-reused 12599 (from 3)
Receiving objects: 100% (12804/12804), 46.51 MiB | 37.39 MiB/s, done.
Resolving deltas: 100% (8066/8066), done.


In [4]:
%cd ragas
!pip install -e . -e ./examples

/content/ragas
Obtaining file:///content/ragas
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Obtaining file:///content/ragas/examples
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ragas (pyproject.toml) ... done
  Created wheel for ragas: filename=ragas-0.4.4.dev8+g298b68274-0.editable-py3-none-any.whl size=13353 sha256=7203a93da3dfa3ce1faf13db4ced2cd8126d5f8b9cb60530815fa99e05e0d843
  Stored in directory: /tmp/pip-ephem-wheel-cache-i9nccqgo/wheels/8c/b7/75/727c7e0e7207f46773c9ca04e1abb11ddba7e0ba53e63ce785
  Building editable for ragas-examples (pyproject.toml) ... done
  Created wheel for ragas-examples: filename=ragas_examples-0.4.4.dev8+g29

In [6]:
!python -m ragas_examples.prompt_evals.evals

/usr/local/lib/python3.12/dist-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/ragas/examples/ragas_examples/prompt_evals/evals.py", line 5, in <module>
    from .prompt import run_prompt
  File "/content/ragas/examples/ragas_examples/prompt_evals/prompt.py", line 5, in <module>
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
                            ~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "<frozen os>", line 714, in __getitem__
KeyError: 'OPE